# 02 · Unión espacial (space join): incidencias × catastro técnico

Parte de las **incidencias ya deduplicadas y geocodificadas** que produce el
notebook `01_preparacion_tickets.ipynb` (`data/processed/eventos.parquet`) y las
cruza con el catastro **por ubicación**. Tres space joins (lógica en
`src/epsel/spatial.py`):

1. ¿En qué **manzana** cae? → punto en polígono (`join_polygon`, `within`).
2. ¿Cuál es la **tubería** más cercana? → vecino más cercano (`join_nearest`).
3. ¿Cuál es el **buzón** más cercano? → vecino más cercano.

Las distancias se miden en UTM 17S (metros). **Salida:**
`data/processed/incidencias_catastro.parquet`.

In [1]:
# Recarga automática de epsel al editarlo
%load_ext autoreload
%autoreload 2
import pandas as pd

from epsel import config, io, spatial

## 1 · Cargar las incidencias geocodificadas (salida del nb01)

`eventos.parquet` ya viene colapsado (1 fila por `EVENTO_ID`) y con coordenadas.

In [2]:
eventos = pd.read_parquet(config.PROCESSED / "eventos.parquet")
print("Incidencias:", eventos.shape)
print("Con coordenada válida:", int(eventos["COORD_VALIDA"].sum()))
eventos.head(3)

Incidencias: (14555, 14)
Con coordenada válida: 10531


,EVENTO_ID,TICKET,N_RECLAMOS,SUMINISTRO,DIRECCION,DISTRITO,TIPO_GRUPO,CATEGORIA_INCIDENCIA,FECHA_PRIMER_RECLAMO,FECHA_ULTIMO_RECLAMO,DURACION_MEDIA_DIAS,LATITUD,LONGITUD,COORD_VALIDA
0,6833976|AGUA|1,11970,6,6833976.0,SANTA ROSA PNP SIN NOMBRE MNZ. 19 LT. 16 NRO.,CHICLAYO,AGUA,FALTA_AGUA,2025-02-03 12:10:02,2025-02-18 07:31:15,24.460581,-6.875963,-79.922069,True
1,1210993|DESAGUE|2,20456,5,1210993.0,SAN MARTIN LA UNION NRO. 156,CHICLAYO,DESAGUE,OTROS,2025-08-15 10:56:59,2025-09-02 17:33:58,12.524336,-6.778739,-79.837919,True
2,1270860|DESAGUE|1,10723,4,1270860.0,SAN ANTONIO SAN CRISTOBAL - P.J.SAN ANTONIO N...,CHICLAYO,DESAGUE,ATORO,2025-01-08 13:55:24,2025-01-22 08:00:45,11.012274,-6.765672,-79.875536,True


## 2 · Convertir a puntos geográficos

`spatial.tickets_to_gdf` conserva solo las incidencias con coordenada válida y
arma un GeoDataFrame de puntos en WGS84.

In [3]:
gdf = spatial.tickets_to_gdf(eventos)
print(f"Incidencias mapeables: {len(gdf):,} ({100*len(gdf)/len(eventos):.1f}%)")
print("CRS:", gdf.crs, "| geometría:", gdf.geom_type.iloc[0])
gdf[["EVENTO_ID", "N_RECLAMOS", "geometry"]].head(3)

Incidencias mapeables: 10,531 (72.4%)
CRS: EPSG:4326 | geometría: Point


,EVENTO_ID,N_RECLAMOS,geometry
0,6833976|AGUA|1,6,POINT (-79.92207 -6.87596)
1,1210993|DESAGUE|2,5,POINT (-79.83792 -6.77874)
2,1270860|DESAGUE|1,4,POINT (-79.87554 -6.76567)


## 3 · Cargar las capas de catastro

Vienen en Web Mercator (EPSG:3857); `spatial` las reproyecta a UTM 17S por dentro.

In [4]:
manzanas = io.load_layer("manzanas")
buzones = io.load_layer("buzones")
redes_sec = io.load_layer("redes_secundarias")
for nombre, capa in [("manzanas", manzanas), ("buzones", buzones), ("redes_sec", redes_sec)]:
    print(f"{nombre:10s}: {len(capa):>6,} features | {capa.geom_type.iloc[0]:12s} | CRS {capa.crs}")

manzanas  :    776 features | Polygon      | CRS EPSG:3857
buzones   : 21,354 features | Point        | CRS EPSG:3857
redes_sec : 29,022 features | LineString   | CRS EPSG:3857


## 4 · Join #1 — ¿en qué manzana cae? (punto en polígono)

In [5]:
gdf = spatial.join_polygon(gdf, manzanas, ["sector", "manzana"])
en_manzana = gdf["manzana"].notna().sum()
print(f"Dentro de una manzana: {en_manzana:,} ({100*en_manzana/len(gdf):.1f}%)")
print("→ Bajo porque el catastro solo entregó el Sector 09 de Chiclayo.")

Dentro de una manzana: 641 (6.1%)
→ Bajo porque el catastro solo entregó el Sector 09 de Chiclayo.


## 5 · Join #2 — ¿tubería más cercana? (vecino más cercano)

La distancia mediana valida la geocodificación: si los puntos cayeran mal, la
tubería "más cercana" estaría a cientos de metros.

In [6]:
gdf = spatial.join_nearest(
    gdf, redes_sec, ["facilityid", "diameter", "material", "networktyp"],
    prefix="red", max_distance=300,
)
print("Distancia a la tubería más cercana (m):")
print(gdf["red_dist_m"].describe().round(1))

Distancia a la tubería más cercana (m):
count    10221.0
mean        12.0
std         19.8
min          0.0
25%          3.3
50%          8.4
75%         14.6
max        267.3
Name: red_dist_m, dtype: float64


## 6 · Join #3 — ¿buzón más cercano?

In [7]:
gdf = spatial.join_nearest(
    gdf, buzones, ["facilityid", "mhtype", "invert", "rimelev"],
    prefix="bz", max_distance=300,
)
print("Distancia al buzón más cercano (m):")
print(gdf["bz_dist_m"].describe().round(1))

Distancia al buzón más cercano (m):
count    10178.0
mean        19.9
std         22.6
min          0.3
25%          7.5
50%         17.0
75%         24.9
max        275.4
Name: bz_dist_m, dtype: float64


## 7 · Revisar una incidencia con todo su cruce

In [8]:
cols = ["EVENTO_ID", "N_RECLAMOS", "CATEGORIA_INCIDENCIA", "manzana", "sector",
        "red_diameter", "red_material", "red_dist_m", "bz_facilityid", "bz_dist_m"]
gdf.sort_values("N_RECLAMOS", ascending=False)[cols].head(3)

,EVENTO_ID,N_RECLAMOS,CATEGORIA_INCIDENCIA,manzana,sector,red_diameter,red_material,red_dist_m,bz_facilityid,bz_dist_m
0,6833976|AGUA|1,6,FALTA_AGUA,NaN,NaN,300.0,CSN,65.394846,FID_162033,50.058576
1,1210993|DESAGUE|2,5,OTROS,NaN,NaN,200.0,CSN,12.141127,FID_097958_0,30.120319
2,1270860|DESAGUE|1,4,ATORO,NaN,NaN,200.0,CSN,4.225189,FID_110574_0,9.752954


## 8 · Guardar (GeoParquet)

In [9]:
out = config.PROCESSED / "incidencias_catastro.parquet"
gdf.to_parquet(out, index=False)
con_red = gdf["red_facilityid"].notna().sum()
print(f"Guardado {out.name}: {gdf.shape}")
print(f"Con tubería atribuida (<300m): {con_red:,} ({100*con_red/len(gdf):.1f}%)")

Guardado incidencias_catastro.parquet: (10531, 27)
Con tubería atribuida (<300m): 10,155 (96.4%)
